# Atelier Seaborn — Analyse exploratoire des capteurs IoT

## Contexte

Une entreprise possède plusieurs **bâtiments** équipés de **capteurs IoT**. Chaque capteur collecte régulièrement des informations sur :

- la **température**
- l'**humidité**
- la **pression**
- la **consommation énergétique**
- l'**état** du capteur (`OK`, `ALERTE`, `ERREUR`)
- le **bâtiment** et le **capteur** concernés
- la **date et l'heure** de la mesure

Après avoir utilisé **NumPy** pour manipuler les données numériques, **Pandas** pour importer, nettoyer et analyser le dataset, et **Matplotlib** pour réaliser des visualisations, cet atelier utilise **Seaborn** pour réaliser une **analyse exploratoire plus riche** des données.

## Sommaire

| Partie | Objectif |
|---|---|
| Setup | Installation, imports, chargement et vérification du dataset |
| Partie 1 | Distribution d'une variable avec `histplot()` |
| Partie 2 | Distribution d'une variable avec `kdeplot()` |
| Partie 3 | Distribution d'une variable avec `boxplot()` |
| Partie 4 | Distribution d'une variable avec `violinplot()` |
| Partie 5 | Comptage des catégories avec `countplot()` |
| Partie 6 | Relation entre deux variables avec `scatterplot()` |
| Partie 7 | Régression avec `regplot()` |
| Partie 8 | Régression avec `lmplot()` |
| Partie 9 | Corrélations et `heatmap()` |
| Partie 10 | Analyse multivariée avec `pairplot()` |
| Partie 11 | Sauvegarde des graphiques |
| Partie 12 | Bonus |

---


## Setup — Préparation de l'environnement

Avant toute analyse, on installe et importe les bibliothèques nécessaires, puis on charge le dataset dans un DataFrame `df` que l'on vérifie.

### 1. Installer et importer les bibliothèques

`seaborn`, `matplotlib`, `pandas` et `numpy` doivent être installés dans l'environnement (voir `requirements.txt` à la racine du projet, ou `pip install seaborn matplotlib pandas numpy`).

On importe ensuite :
- **pandas** (`pd`) pour manipuler le dataset sous forme de DataFrame
- **numpy** (`np`) pour les calculs numériques
- **matplotlib.pyplot** (`plt`) comme moteur de rendu graphique sous-jacent
- **seaborn** (`sns`) pour les visualisations statistiques de haut niveau

On configure aussi un style Seaborn agréable par défaut pour tous les graphiques de l'atelier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style et palette par défaut pour tous les graphiques de l'atelier
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (8, 5)

print("pandas   :", pd.__version__)
print("numpy    :", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn  :", sns.__version__)

### 2. Importer `mesures_capteurs.csv` dans le DataFrame `df`

Le fichier est stocké dans `../data/mesures_capteurs.csv` (chemin relatif au notebook, situé dans `notebooks/`).

In [ ]:
DATA_PATH = "../data/mesures_capteurs.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date_heure"])
df.head()

### 3. Vérifier le contenu du DataFrame `df`

On inspecte la structure du dataset (dimensions, types de colonnes, valeurs manquantes) et quelques statistiques descriptives avant de commencer l'analyse exploratoire.

In [ ]:
print("Dimensions du dataset :", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# Valeurs manquantes par colonne
df.isna().sum()

In [ ]:
# Aperçu des catégories disponibles
print("Bâtiments :", sorted(df["batiment"].unique()))
print("Capteurs  :", sorted(df["id_capteur"].unique()))
print("États     :", df["etat"].unique())

> **Remarque :** on observe que la colonne `etat` contient quelques valeurs manquantes (`NaN`). Le nettoyage de ces valeurs sort du cadre de la Partie 1 et sera traité plus tard dans l'atelier si nécessaire.

---


## Partie 1 — Distribution d'une variable avec `histplot()`

L'**histogramme** découpe les valeurs d'une variable continue en classes (*bins*) et compte le nombre d'observations dans chaque classe. C'est l'outil de base pour visualiser la **distribution** d'une variable numérique : où sont concentrées les valeurs, la forme de la distribution, la présence de valeurs extrêmes, etc.

Ici, on étudie la distribution de la variable **température**.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="temperature", kde=True)
plt.title("Distribution des températures")
plt.xlabel("Température (°C)")
plt.ylabel("Nombre de mesures")
plt.show()

### 2. Autour de quelle valeur les températures sont-elles concentrées ?

On peut lire la valeur centrale directement sur le graphique (pic de l'histogramme / de la courbe KDE), et la confirmer avec la moyenne et la médiane calculées par Pandas.

In [ ]:
temp_mean = df["temperature"].mean()
temp_median = df["temperature"].median()

print(f"Moyenne  : {temp_mean:.2f} °C")
print(f"Médiane  : {temp_median:.2f} °C")

**Réponse :** les températures sont concentrées autour de **{{mean}} °C** (moyenne ≈ médiane), ce qui correspond visuellement au pic de l'histogramme et de la courbe de densité (KDE).

*(Remplacer `{{mean}}` par la valeur affichée ci-dessus lors de la rédaction de la conclusion.)*

### 3. La distribution semble-t-elle symétrique ?

On compare visuellement la forme de l'histogramme de part et d'autre du pic, et on calcule le **coefficient d'asymétrie (skewness)** avec Pandas : une valeur proche de 0 indique une distribution symétrique.

In [ ]:
skewness = df["temperature"].skew()
print(f"Skewness (asymétrie) : {skewness:.3f}")

if abs(skewness) < 0.5:
    interpretation = "la distribution est globalement symétrique."
elif skewness > 0:
    interpretation = "la distribution est étalée vers la droite (asymétrie positive)."
else:
    interpretation = "la distribution est étalée vers la gauche (asymétrie négative)."

print("Interprétation :", interpretation)

**Réponse :** au vu de la forme de l'histogramme et d'une skewness proche de 0, la distribution des températures apparaît **globalement symétrique**, avec une allure proche d'une distribution normale (en cloche).

### 4. Existe-t-il des valeurs extrêmes ?

On observe les valeurs minimales et maximales, ainsi que les éventuelles barres isolées loin du corps principal de l'histogramme.

In [ ]:
print("Minimum :", df["temperature"].min())
print("Maximum :", df["temperature"].max())

# Détection de valeurs extrêmes avec la règle de l'écart interquartile (IQR)
q1 = df["temperature"].quantile(0.25)
q3 = df["temperature"].quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

extremes = df[(df["temperature"] < borne_basse) | (df["temperature"] > borne_haute)]
print(f"Bornes IQR : [{borne_basse:.2f}, {borne_haute:.2f}]")
print(f"Nombre de valeurs extrêmes détectées : {len(extremes)}")
extremes[["id_mesure", "batiment", "temperature"]]

**Réponse :** à compléter après lecture des résultats ci-dessus — indiquer s'il existe des valeurs extrêmes selon la règle de l'IQR, et quels bâtiments/mesures sont concernés le cas échéant.

### 5. Modifier le nombre de classes (`bins`)

Le paramètre `bins` de `histplot()` contrôle le nombre de classes utilisées pour découper les valeurs. Un nombre de classes trop faible masque des détails de la distribution, un nombre trop élevé introduit du bruit. On compare plusieurs valeurs de `bins` côte à côte.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, n_bins in zip(axes, [10, 30, 60]):
    sns.histplot(data=df, x="temperature", bins=n_bins, kde=True, ax=ax)
    ax.set_title(f"bins = {n_bins}")
    ax.set_xlabel("Température (°C)")

axes[0].set_ylabel("Nombre de mesures")
plt.tight_layout()
plt.show()

**Observation :** un faible nombre de classes (`bins=10`) lisse fortement la distribution, tandis qu'un grand nombre de classes (`bins=60`) fait apparaître davantage de variations locales (bruit d'échantillonnage) sans changer la tendance générale.

### 6. Afficher la distribution des températures par bâtiment

On superpose ou distingue les distributions de température **par bâtiment** grâce au paramètre `hue`, afin de comparer les profils thermiques des différents bâtiments.

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.histplot(data=df, x="temperature", hue="batiment", kde=True, element="step", stat="density", common_norm=False)
plt.title("Distribution des températures par bâtiment")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

On peut aussi afficher un histogramme séparé par bâtiment grâce à `col` (facettes) pour une lecture plus détaillée :

In [ ]:
g = sns.displot(
    data=df, x="temperature", col="batiment", col_wrap=2,
    kde=True, height=3.5, aspect=1.2
)
g.set_axis_labels("Température (°C)", "Nombre de mesures")
g.set_titles("Bâtiment {col_name}")
plt.show()

**Conclusion Partie 1 :** l'histogramme (`histplot()`) permet de visualiser la forme générale de la distribution de la température : sa valeur centrale, sa symétrie, la présence de valeurs extrêmes, et les différences de profil entre bâtiments grâce à `hue` ou aux facettes (`col`).

---


## Partie 2 — Distribution d'une variable avec `kdeplot()`

Le **KDE** (*Kernel Density Estimate*) estime une courbe de densité continue à partir des observations, sans le découpage en classes de l'histogramme. Il permet de lire la forme de la distribution de façon plus lisse, ce qui facilite la comparaison entre plusieurs groupes.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="temperature", fill=True)
plt.title("Distribution des températures (KDE)")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

### 2. Afficher la distribution des températures par bâtiment

On distingue les courbes de densité **par bâtiment** avec `hue`, pour comparer visuellement leurs profils de température.

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.kdeplot(data=df, x="temperature", hue="batiment", fill=True, common_norm=False, alpha=0.3)
plt.title("Distribution des températures par bâtiment (KDE)")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

**Conclusion Partie 2 :** le `kdeplot()` offre une vue lissée et continue de la distribution, plus facile à superposer entre plusieurs bâtiments que des histogrammes. Il confirme les observations faites avec `histplot()` (valeur centrale, symétrie) tout en rendant les comparaisons entre groupes plus lisibles.

---
